## Лабораторная 4. Ансамбли, полносвязные нейронные сети

**Цель работы:** решить задачу классификации кредитоспособности, используя ансамблевые методы и многослойные перцептроны.

**План выполнения:**
1. Загрузка и первичный анализ данных
2. Предобработка и разделение выборки
3. Обучение базовых моделей (baseline)
4. Оптимизация гиперпараметров через GridSearchCV
5. Сравнительный анализ метрик на тестовой выборке

**Критерии оценки (ROC-AUC на тесте):**
- ≤ 0.76 → 0 баллов
- 0.76–0.77 → 2 балла
- 0.77–0.78 → 4 балла
- 0.78–0.79 → 6 баллов
- 0.79–0.80 → 8 баллов
- > 0.80 → 10 баллов

In [ ]:
# Импорт необходимых библиотек
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, 
    recall_score, classification_report, confusion_matrix
)

# Настройки визуализации и воспроизводимости
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

In [ ]:
# === Шаг 1: Загрузка данных ===
df = pd.read_csv('german.csv', sep=';')

print(f"📊 Размер датасета: {df.shape}")
print(f"📋 Столбцы: {list(df.columns)}")
print("\n🔍 Первые 5 строк:")
display(df.head())

# Проверка на пропуски и типы данных
print(f"\n✅ Пропущенные значения: {df.isnull().sum().sum()}")
print(f"📈 Типы данных:\n{df.dtypes.value_counts()}")

In [ ]:
# === Шаг 2: Разведочный анализ (EDA) ===
target_col = 'Creditability'
feature_cols = df.columns.drop(target_col)

# Распределение целевой переменной
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Гистограмма классов
ax[0].bar([0, 1], df[target_col].value_counts().sort_index(), edgecolor='black', color=['#ff6b6b', '#4ecdc4'])
ax[0].set_xticks([0, 1])
ax[0].set_xlabel('Класс (0: ненадёжный, 1: надёжный)')
ax[0].set_ylabel('Количество')
ax[0].set_title('Распределение целевой переменной')

# Корреляционная матрица (топ-10 признаков)
corr = df.corr().abs()
top_corr = corr.nlargest(10, target_col)[target_col].index
sns.heatmap(df[top_corr].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax[1])
ax[1].set_title('Корреляция топ-10 признаков с целевой')

plt.tight_layout()
plt.show()

# Статистика по классам
print(f"\n📊 Доля надёжных клиентов: {df[target_col].mean():.2%}")
print(f"📊 Дисбаланс классов: {df[target_col].value_counts(normalize=True).to_dict()}")

In [ ]:
# === Шаг 3: Подготовка данных ===
# Разделение на признаки и целевую переменную
X_raw = df.drop(columns=[target_col]).values
y_raw = df[target_col].values

# Стратифицированное разделение на обучающую и тестовую выборки (80/20)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_raw, y_raw, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_raw
)

print(f"✅ Обучающая выборка: {X_tr.shape}, Тестовая: {X_te.shape}")
print(f"✅ Распределение классов в train: {np.bincount(y_tr)}")
print(f"✅ Распределение классов в test: {np.bincount(y_te)}")

# Масштабирование для нейросети (стандартизация)
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_te_scaled = scaler.transform(X_te)

In [ ]:
# === Шаг 4: Обучение базовых моделей (baseline) ===

def evaluate_clf(model, X_train, X_test, y_train, y_test, name, use_scaled=False):
    """Универсальная функция оценки классификатора"""
    # Выбор данных (масштабированные или исходные)
    X_tr_use = X_train if not use_scaled else scaler.transform(X_train)
    X_te_use = X_test if not use_scaled else scaler.transform(X_test)
    
    # Обучение модели
    model.fit(X_tr_use, y_train)
    
    # Предсказания (вероятности для ROC-AUC)
    if hasattr(model, 'predict_proba'):
        y_pred_prob = model.predict_proba(X_te_use)[:, 1]
    else:
        # Для моделей без predict_proba используем decision_function
        scores = model.decision_function(X_te_use)
        y_pred_prob = (scores - scores.min()) / (scores.max() - scores.min())
    
    y_pred = model.predict(X_te_use)
    
    # Расчёт метрик
    metrics = {
        'ROC-AUC': roc_auc_score(y_test, y_pred_prob),
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred)
    }
    
    # Вывод результатов
    print(f"\n🔹 {name}:")
    for k, v in metrics.items():
        print(f"   {k}: {v:.4f}")
    
    return metrics, model

# 4.1 Random Forest (baseline)
rf_base = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_metrics, rf_model_bl = evaluate_clf(rf_base, X_tr, X_te, y_tr, y_te, "Random Forest (baseline)")

# 4.2 Gradient Boosting (baseline)
gb_base = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_metrics, gb_model_bl = evaluate_clf(gb_base, X_tr, X_te, y_tr, y_te, "Gradient Boosting (baseline)")

# 4.3 MLP Neural Network (baseline)
mlp_base = MLPClassifier(hidden_layer_sizes=(30,), max_iter=500, random_state=42, early_stopping=True)
mlp_metrics, mlp_model_bl = evaluate_clf(
    mlp_base, X_tr_scaled, X_te_scaled, y_tr, y_te, 
    "MLP Neural Network (baseline)", use_scaled=True
)

In [ ]:
# === Шаг 5: Оптимизация гиперпараметров ===

print("🚀 Запуск подбора гиперпараметров...\n")

# 5.1 Random Forest — расширенный поиск
print("🔍 Оптимизация Random Forest...")
rf_grid_params = {
    'n_estimators': [150, 200, 250],
    'max_depth': [8, 12, 16, None],
    'min_samples_split': [3, 5, 7],
    'min_samples_leaf': [1, 2, 3],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced', None]
}

rf_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_grid_params,
    cv=5,
    scoring='roc_auc',
    verbose=0,
    n_jobs=-1
)
rf_search.fit(X_tr, y_tr)
print(f"✅ Лучшие параметры RF: {rf_search.best_params_}")
rf_opt_metrics, rf_model_opt = evaluate_clf(
    rf_search.best_estimator_, X_tr, X_te, y_tr, y_te, "Random Forest (optimized)"
)

# 5.2 Gradient Boosting — тонкая настройка
print("\n🔍 Оптимизация Gradient Boosting...")
gb_grid_params = {
    'n_estimators': [150, 200],
    'learning_rate': [0.03, 0.05, 0.07],
    'max_depth': [3, 4, 5],
    'min_samples_split': [2, 4],
    'min_samples_leaf': [1, 2],
    'subsample': [0.8, 0.9],
    'max_features': ['sqrt', 'log2']
}

gb_search = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    gb_grid_params,
    cv=5,
    scoring='roc_auc',
    verbose=0,
    n_jobs=-1
)
gb_search.fit(X_tr, y_tr)
print(f"✅ Лучшие параметры GB: {gb_search.best_params_}")
gb_opt_metrics, gb_model_opt = evaluate_clf(
    gb_search.best_estimator_, X_tr, X_te, y_tr, y_te, "Gradient Boosting (optimized)"
)

# 5.3 MLP — архитектура и регуляризация
print("\n🔍 Оптимизация MLP Neural Network...")
mlp_grid_params = {
    'hidden_layer_sizes': [(64,), (100,), (64, 32), (128, 64)],
    'activation': ['relu', 'tanh'],
    'alpha': [1e-4, 1e-3, 1e-2],  # L2-регуляризация
    'learning_rate_init': [0.001, 0.005],
    'max_iter': [800, 1000],
    'early_stopping': [True]
}

mlp_search = GridSearchCV(
    MLPClassifier(random_state=42, early_stopping=True),
    mlp_grid_params,
    cv=5,
    scoring='roc_auc',
    verbose=0,
    n_jobs=-1
)
mlp_search.fit(X_tr_scaled, y_tr)
print(f"✅ Лучшие параметры MLP: {mlp_search.best_params_}")
mlp_opt_metrics, mlp_model_opt = evaluate_clf(
    mlp_search.best_estimator_, X_tr_scaled, X_te_scaled, y_tr, y_te, 
    "MLP Neural Network (optimized)", use_scaled=True
)

In [ ]:
# === Шаг 6: Сравнительный анализ результатов ===

# Сбор всех результатов в таблицу
results_df = pd.DataFrame({
    'Model': [
        'RF (baseline)', 'RF (optimized)',
        'GB (baseline)', 'GB (optimized)', 
        'MLP (baseline)', 'MLP (optimized)'
    ],
    'ROC-AUC': [
        rf_metrics['ROC-AUC'], rf_opt_metrics['ROC-AUC'],
        gb_metrics['ROC-AUC'], gb_opt_metrics['ROC-AUC'],
        mlp_metrics['ROC-AUC'], mlp_opt_metrics['ROC-AUC']
    ],
    'Accuracy': [
        rf_metrics['Accuracy'], rf_opt_metrics['Accuracy'],
        gb_metrics['Accuracy'], gb_opt_metrics['Accuracy'],
        mlp_metrics['Accuracy'], mlp_opt_metrics['Accuracy']
    ],
    'Precision': [
        rf_metrics['Precision'], rf_opt_metrics['Precision'],
        gb_metrics['Precision'], gb_opt_metrics['Precision'],
        mlp_metrics['Precision'], mlp_opt_metrics['Precision']
    ],
    'Recall': [
        rf_metrics['Recall'], rf_opt_metrics['Recall'],
        gb_metrics['Recall'], gb_opt_metrics['Recall'],
        mlp_metrics['Recall'], mlp_opt_metrics['Recall']
    ]
})

# Сортировка по ROC-AUC (убывание)
results_df = results_df.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

# Визуализация сравнения метрик
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График ROC-AUC по моделям
colors = ['#ff6b6b' if 'baseline' in m else '#4ecdc4' for m in results_df['Model']]
bars = axes[0].barh(results_df['Model'], results_df['ROC-AUC'], color=colors, edgecolor='black')
axes[0].set_xlabel('ROC-AUC Score')
axes[0].set_title('Сравнение моделей по ROC-AUC')
axes[0].axvline(x=0.80, color='gold', linestyle='--', label='Цель: > 0.80')
axes[0].legend()

# Тепловая карта всех метрик
metrics_to_plot = ['ROC-AUC', 'Accuracy', 'Precision', 'Recall']
heatmap_data = results_df.set_index('Model')[metrics_to_plot].T
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[1])
axes[1].set_title('Все метрики по моделям')

plt.tight_layout()
plt.show()

# Вывод итоговой таблицы
print("\n🏆 ИТОГОВАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ (отсортировано по ROC-AUC):")
display(results_df.style.format('{:.4f}').background_gradient(cmap='Greens', subset=['ROC-AUC']))

# Лучшая модель и её оценка по критериям лабы
best_auc = results_df.iloc[0]['ROC-AUC']
best_model = results_df.iloc[0]['Model']

if best_auc > 0.80:
    score = 10
elif best_auc > 0.79:
    score = 8
elif best_auc > 0.78:
    score = 6
elif best_auc > 0.77:
    score = 4
elif best_auc > 0.76:
    score = 2
else:
    score = 0

print(f"\n🎯 Лучшая модель: {best_model}")
print(f"🎯 ROC-AUC: {best_auc:.4f}")
print(f"🎯 Предполагаемая оценка: {score}/10 баллов")

In [ ]:
# === Дополнительно: Анализ важности признаков (для лучшей модели) ===

if 'RF' in best_model or 'GB' in best_model:
    # Для ансамблей можно посмотреть feature_importances_
    if 'RF' in best_model:
        best_model_obj = rf_model_opt
    else:
        best_model_obj = gb_model_opt
    
    # Топ-10 наиболее важных признаков
    importances = best_model_obj.feature_importances_
    indices = np.argsort(importances)[::-1][:10]
    
    plt.figure(figsize=(10, 6))
    plt.barh(range(10), importances[indices][::-1], color='#6c5ce7', edgecolor='black')
    plt.yticks(range(10), [feature_cols[i] for i in indices[::-1]])
    plt.xlabel('Важность признака')
    plt.title('Топ-10 наиболее значимых признаков')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print("\n📌 Топ-5 признаков для прогноза:")
    for i in range(5):
        print(f"   {i+1}. {feature_cols[indices[i]]}: {importances[indices[i]]:.4f}")

# === Кривая обучения для лучшей модели (опционально) ===
print("\n💡 Рекомендации для дальнейшего улучшения:")
print("   • Попробовать ансамбль из нескольких моделей (Voting/Stacking)")
print("   • Добавить feature engineering (полиномиальные признаки, взаимодействия)")
print("   • Использовать более продвинутые методы обработки дисбаланса (SMOTE, focal loss)")
print("   • Экспериментировать с архитектурой нейросети (больше слоёв, dropout, batch normalization)")